In [1]:
%%capture
%pip install --upgrade pip
%pip install "Pillow==9.5.0" wordcloud nltk urllib3~=2.0
!pip uninstall xgboost -y
!pip install "xgboost==1.7.6" --no-cache-dir

In [2]:
!pip install pandas numpy matplotlib seaborn tqdm scikit-learn nltk xgboost Pillow wordcloud


Defaulting to user installation because normal site-packages is not writeable


In [3]:
%%capture
from tqdm import tqdm
import numpy as np
import pandas as pd
from itertools import accumulate
import matplotlib.pyplot as plt
import seaborn as sns
import multiprocessing
%matplotlib inline

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from scipy.stats import uniform, loguniform


import string
import re

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.tokenize import RegexpTokenizer
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.corpus import wordnet
nltk.download('vader_lexicon')
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')

####
from xgboost import XGBClassifier
import xgboost as xgb

####
from PIL import Image
from wordcloud import WordCloud, STOPWORDS, ImageColorGenerator

# You can also use this section to suppress warnings generated by your code:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

sns.set_context('notebook')
sns.set_style('white')

In [4]:
tweets = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMSkillsNetwork-GPXX0DYDEN/twitterv3.csv')

In [6]:
tweets.head(3)


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2,3.0
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,1,4.0


In [8]:
tweets.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1372 entries, 0 to 1371
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   tweet_id                 1372 non-null   int64  
 1   author_id                1372 non-null   object 
 2   inbound                  1372 non-null   bool   
 3   created_at               1372 non-null   object 
 4   text                     1372 non-null   object 
 5   response_tweet_id        931 non-null    object 
 6   in_response_to_tweet_id  1022 non-null   float64
dtypes: bool(1), float64(1), int64(1), object(4)
memory usage: 65.8+ KB


In [14]:
df = tweets
df['new_text'] = df['text'].str.lower()
df.sample(3)

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id,new_text
371,665,AmazonHelp,False,Tue Oct 31 22:19:29 +0000 2017,@115843 You're not lion! She even shows witch ...,666,667.0,@115843 you're not lion! she even shows witch ...
206,295,115780,True,Tue Oct 31 17:01:38 +0000 2017,@115777 your online forum is always broken eve...,294,NaN,@115777 your online forum is always broken eve...
376,670,115844,True,Tue Oct 31 22:15:42 +0000 2017,@115845 They sent the feckin' order to Germany...,668,671.0,@115845 they sent the feckin' order to germany...


In [9]:
nltk.download('punkt_tab')

True

In [10]:
def negate(sentence):
    temp = int(0)
    for i in range(len(sentence)):
        if sentence[i-1] in ['not',"n't"]:
            antonyms = []
            for syn in wordnet.synsets(sentence[i]):
                syns = wordnet.synsets(sentence[i])
                w1 = syns[0].name()
                temp = 0
                for l in syn.lemmas():
                    if l.antonyms():
                        antonyms.append(l.antonyms()[0].name())
                        max_dissimilarity = 0
                        for ant in antonyms:
                            syns = wordnet.synsets(ant)
                            w2 = syns[0].name()
                            syns = wordnet.synsets(sentence[i])
                            w1 = syns[0].name()
                            word1 = wordnet.synset(w1)
                            word2 = wordnet.synset(w2)
                            if isinstance(word1.wup_similarity(word2), float) or isinstance(word1.wup_similarity(word2), int):
                                temp = 1 - word1.wup_similarity(word2)
                                if temp>max_dissimilarity:
                                    max_dissimilarity = temp
                                    antonym_max = ant
                                    sentence[i] = antonym_max
                                    sentence[i-1] = ''
    while '' in sentence:
        sentence.remove('')
    return sentence

In [13]:
sentence = word_tokenize("he isn't happy")
print(negate(sentence))

['he', 'is', 'unhappy']


In [15]:
df['new_text'] = df['new_text'].apply(lambda x: negate(word_tokenize(x)))
df.head(5)

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id,new_text
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2,3.0,"[@, 115712, i, understand, ., i, would, like, ..."
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0,"[@, sprintcare, and, how, do, you, propose, we..."
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,1,4.0,"[@, sprintcare, i, have, sent, several, privat..."
3,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so th...,3,5.0,"[@, 115712, please, send, us, a, private, mess..."
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4,6.0,"[@, sprintcare, i, did, .]"


In [16]:
pd.set_option('display.max_colwidth', None)
df[['text', 'new_text']].head(5)

,text,new_text
0,@115712 I understand. I would like to assist you. We would need to get you into a private secured link to further assist.,"[@, 115712, i, understand, ., i, would, like, to, assist, you, ., we, would, need, to, get, you, into, a, private, secured, link, to, further, assist, .]"
1,@sprintcare and how do you propose we do that,"[@, sprintcare, and, how, do, you, propose, we, do, that]"
2,@sprintcare I have sent several private messages and no one is responding as usual,"[@, sprintcare, i, have, sent, several, private, messages, and, no, one, is, responding, as, usual]"
3,@115712 Please send us a Private Message so that we can further assist you. Just click ‘Message’ at the top of your profile.,"[@, 115712, please, send, us, a, private, message, so, that, we, can, further, assist, you, ., just, click, ‘, message, ’, at, the, top, of, your, profile, .]"
4,@sprintcare I did.,"[@, sprintcare, i, did, .]"


In [17]:
def unlist(list):
    words=''
    for item in list:
        words+=item+' '
    return words

df['new_text'] = df['new_text'].apply(lambda x: unlist(x))
df.head()

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id,new_text
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist you. We would need to get you into a private secured link to further assist.,2,3.0,@ 115712 i understand . i would like to assist you . we would need to get you into a private secured link to further assist .
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0,@ sprintcare and how do you propose we do that
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messages and no one is responding as usual,1,4.0,@ sprintcare i have sent several private messages and no one is responding as usual
3,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so that we can further assist you. Just click ‘Message’ at the top of your profile.,3,5.0,@ 115712 please send us a private message so that we can further assist you . just click ‘ message ’ at the top of your profile .
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4,6.0,@ sprintcare i did .


In [19]:
punc = string.punctuation
print("The punctuation signs in Python are: " + punc)

The punctuation signs in Python are: !"#$%&'()*+,-./:;<=>?@[\]^_`{|}~


In [20]:
def delete_punc(text):
    text_wo_punc = text.translate(str.maketrans("", "", punc))
    return text_wo_punc

df['new_text'] = df['new_text'].apply(lambda text: delete_punc(text))
df.head(20)

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id,new_text
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist you. We would need to get you into a private secured link to further assist.,2,3.0,115712 i understand i would like to assist you we would need to get you into a private secured link to further assist
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0,sprintcare and how do you propose we do that
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messages and no one is responding as usual,1,4.0,sprintcare i have sent several private messages and no one is responding as usual
3,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so that we can further assist you. Just click ‘Message’ at the top of your profile.,3,5.0,115712 please send us a private message so that we can further assist you just click ‘ message ’ at the top of your profile
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4,6.0,sprintcare i did
5,6,sprintcare,False,Tue Oct 31 21:46:24 +0000 2017,"@115712 Can you please send us a private message, so that I can gain further details about your account?","5,7",8.0,115712 can you please send us a private message so that i can gain further details about your account
6,8,115712,True,Tue Oct 31 21:45:10 +0000 2017,@sprintcare is the worst customer service,"9,6,10",NaN,sprintcare is the worst customer service
7,11,sprintcare,False,Tue Oct 31 22:10:35 +0000 2017,"@115713 This is saddening to hear. Please shoot us a DM, so that we can look into this for you. -KC",NaN,12.0,115713 this is saddening to hear please shoot us a dm so that we can look into this for you kc
8,12,115713,True,Tue Oct 31 22:04:47 +0000 2017,@sprintcare You gonna magically change your connectivity for me and my whole family ? 🤥 💯,"11,13,14",15.0,sprintcare you gon na magically change your connectivity for me and my whole family 🤥 💯
9,15,sprintcare,False,Tue Oct 31 20:03:31 +0000 2017,"@115713 We understand your concerns and we'd like for you to please send us a Direct Message, so that we can further assist you. -AA",12,16.0,115713 we understand your concerns and we d like for you to please send us a direct message so that we can further assist you aa


In [21]:
print("The stopwords in English are: ")
", ".join(stopwords.words("english"))

The stopwords in English are: 


"a, about, above, after, again, against, ain, all, am, an, and, any, are, aren, aren't, as, at, be, because, been, before, being, below, between, both, but, by, can, couldn, couldn't, d, did, didn, didn't, do, does, doesn, doesn't, doing, don, don't, down, during, each, few, for, from, further, had, hadn, hadn't, has, hasn, hasn't, have, haven, haven't, having, he, he'd, he'll, her, here, hers, herself, he's, him, himself, his, how, i, i'd, if, i'll, i'm, in, into, is, isn, isn't, it, it'd, it'll, it's, its, itself, i've, just, ll, m, ma, me, mightn, mightn't, more, most, mustn, mustn't, my, myself, needn, needn't, no, nor, not, now, o, of, off, on, once, only, or, other, our, ours, ourselves, out, over, own, re, s, same, shan, shan't, she, she'd, she'll, she's, should, shouldn, shouldn't, should've, so, some, such, t, than, that, that'll, the, their, theirs, them, themselves, then, there, these, they, they'd, they'll, they're, they've, this, those, through, to, too, under, until, up, 